# 1. Reproducible synthetic commodity price data

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
np.random.seed(42)

commodities = {
    "Gold": 1900,
    "Crude Oil": 80,
    "Copper": 8500,
    "Soybeans": 1300
}

dates = pd.date_range(
    start="2023-01-01",
    periods=2500,
    freq="h"
)

data = []

for commodity, initial_price in commodities.items():
    returns = np.random.normal(
        loc=0.0001,
        scale=0.01,
        size=len(dates)
    )

    prices = initial_price * np.exp(
        np.cumsum(returns)
    )

    temp = pd.DataFrame({
        "Date": dates,
        "Commodity": commodity,
        "Price": prices
    })

    data.append(temp)

df = pd.concat(data, ignore_index=True)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (10000, 3)
                 Date Commodity        Price
0 2023-01-01 00:00:00      Gold  1909.652002
1 2023-01-01 01:00:00      Gold  1907.204171
2 2023-01-01 02:00:00      Gold  1919.788973
3 2023-01-01 03:00:00      Gold  1949.446661
4 2023-01-01 04:00:00      Gold  1945.081804


# 2. Data Cleaning and Return Calculation



In [3]:
df_clean = df.copy()

print("Missing values:")
print(df_clean.isnull().sum())
print("\nDuplicate records:")
print(df_clean.duplicated(subset=["Date", "Commodity"]).sum())

df_clean = df_clean.drop_duplicates(subset=["Date", "Commodity"])
df_clean = df_clean.dropna(subset=["Date", "Commodity", "Price"])
df_clean = df_clean[df_clean["Price"] > 0].copy()

df_clean = df_clean.sort_values(by=["Commodity", "Date"]).reset_index(drop=True)

df_clean["Log_Return"] = (df_clean.groupby("Commodity")["Price"].transform(lambda x: np.log(x / x.shift(1))))

df_clean = df_clean.dropna(subset=["Log_Return"]).reset_index(drop=True)

print("\nCleaned dataset shape:", df_clean.shape)
print("\nSample observations:")
print(df_clean.head())
print("\nSummary statistics of log returns:")
print(df_clean.groupby("Commodity")["Log_Return"].agg(["mean", "std", "min", "max"]))


Missing values:
Date         0
Commodity    0
Price        0
dtype: int64

Duplicate records:
0

Cleaned dataset shape: (9996, 4)

Sample observations:
                 Date Commodity        Price  Log_Return
0 2023-01-01 01:00:00    Copper  8427.451605   -0.004434
1 2023-01-01 02:00:00    Copper  8278.302985   -0.017856
2 2023-01-01 03:00:00    Copper  8251.847313   -0.003201
3 2023-01-01 04:00:00    Copper  8313.372665    0.007428
4 2023-01-01 05:00:00    Copper  8208.933903   -0.012642

Summary statistics of log returns:
               mean       std       min       max
Commodity                                        
Copper    -0.000185  0.010230 -0.032110  0.035391
Crude Oil -0.000128  0.010095 -0.031667  0.039362
Gold       0.000436  0.009830 -0.032313  0.038627
Soybeans   0.000183  0.009973 -0.039124  0.033874


# 3. Commodity Risk Measurement

In [4]:
confidence_level = 0.95
alpha = 1 - confidence_level

risk_results = []

for commodity, group in df_clean.groupby("Commodity"):

    returns = group["Log_Return"].dropna()

    # （1）Realized volatility over the full sample
    rv = np.sqrt(np.sum(returns ** 2))

    # （2）Historical Value at Risk (95%)
    var_95 = -np.quantile(returns, alpha)

    # （3）Conditional Value at Risk (95%)
    tail_returns = returns[
        returns <= -var_95]

    cvar_95 = -tail_returns.mean()

    risk_results.append({
        "Commodity": commodity,
        "Realized_Volatility": rv,
        "VaR_95": var_95,
        "CVaR_95": cvar_95})

risk_df = pd.DataFrame(risk_results)

print("Commodity Risk Assessment:")
display(risk_df.round(4))

Commodity Risk Assessment:


,Commodity,Realized_Volatility,VaR_95,CVaR_95
0,Copper,0.5114,0.0172,0.0209
1,Crude Oil,0.5046,0.0173,0.0215
2,Gold,0.4918,0.0155,0.0196
3,Soybeans,0.4985,0.0163,0.0204


# 4. Interactive Data Visualization

4.1 Normalized Commodity Price Trends

In [5]:
import plotly.express as px
price_df = df_clean.copy()
price_df["Normalized_Price"] = (
    price_df.groupby("Commodity")["Price"]
    .transform(lambda x: x / x.iloc[0] * 100))

fig1 = px.line(
    price_df,
    x="Date",
    y="Normalized_Price",
    color="Commodity",
    title="Normalized Commodity Price Trends",
    labels={"Date": "Date","Normalized_Price": "Price Index (Base = 100)"})

fig1.update_layout(template="plotly_white", hovermode="x unified",title_x=0.5)
fig1.show()

4.2 VaR and CVaR Comparison

In [6]:
risk_long = risk_df.melt(
    id_vars="Commodity",
    value_vars=["VaR_95", "CVaR_95"],
    var_name="Risk_Metric",
    value_name="Risk_Value"
)

fig2 = px.bar(
    risk_long,
    x="Commodity",
    y="Risk_Value",
    color="Risk_Metric",
    barmode="group",
    title="Commodity Risk Comparison: VaR vs CVaR",
    labels={
        "Risk_Value": "Risk Level",
        "Commodity": "Commodity",
        "Risk_Metric": "Risk Metric"
    },
    text_auto=".2%"
)

fig2.update_layout(
    template="plotly_white",
    title_x=0.5,
    yaxis_tickformat=".1%"
)

fig2.show()


4.3 Rolling Volatility Analysis

In [7]:
vol_df = df_clean.copy()
vol_df["Rolling_Volatility_24H"] = (
    vol_df.groupby("Commodity")["Log_Return"]
    .transform(
        lambda x: x.rolling(
            window=24,
            min_periods=24
        ).std()))

fig3 = px.line(
    vol_df.dropna(subset=["Rolling_Volatility_24H"]),
    x="Date",
    y="Rolling_Volatility_24H",
    color="Commodity",
    title="24-Hour Rolling Volatility by Commodity",
    labels={
        "Date": "Date",
        "Rolling_Volatility_24H": "Rolling Standard Deviation"})

fig3.update_layout(
    template="plotly_white",
    title_x=0.5,
    yaxis_tickformat=".2%")

fig3.show()